# Notebook 32 — Can additive autapses reproduce frozen ESCM 2.0?

## Scientific question

Can the **original non-fading, frozen ESCM 2.0** be reproduced functionally by replacing its recurrent current **assignment** with **additive** autaptic excitation and, if useful, changing the absolute refractory period?

The comparison is deliberately restricted to **Stage A**. There is **no sensory support, no short-term depression, and no sensory-replenished resource (SRR)** in either model. We will not incorporate SRR until the additive-only network passes the benchmarks.

**Frozen reference:** `Iexc_post = 8`, refractory period 0 ms, exactly as in Notebooks 19–20.  
**Candidate:** `Iexc_post += W`, refractory period chosen by calibration.

We preserve the 9×9 grid, original current-based LIF equations, camera pulses, homogeneous surround-inhibitory graph, threshold/reset, and `dt=0.05 ms`. The inhibitory weights and decay constants stay fixed throughout calibration. This tests whether autaptic strength and refractoriness **alone** can recover the frozen network's behavior.

**Success criterion:** reproduce the reference's eligible old-only/new-only controls, its 3/5 strict replacement outcomes (vertical, horizontal, L-shaped), comparable sustained firing in isolated neurons, and stable behavior under the original timing and timestep checks. Functional agreement does not require identical individual spikes or membrane trajectories. If the restricted parameter search fails, report failure; do not silently change inhibition or add SRR.

Original notebooks 20 and 25 are the authoritative sources for the reference, geometry, and scoring conventions. This notebook reuses Notebook 25's simulator directly, with its target explicitly set to **original**.

## 1. Preserve the frozen ESCM 2.0 geometry and inhibitory graph

Each pixel is one neuron. The old and new patterns are **stimuli**, not labels or connection rules: every simulation uses exactly the same distance-based inhibitory graph. For inhibitory distances $d_{ij}$,

$$
w^I_{ij}=0.1\exp\left(-\frac{d_{ij}^2}{2(2.5)^2}\right)
\mathbf{1}\{1.5<d_{ij}\le 5.5\}.
$$

We use five pre-established geometries. **Vertical and horizontal** are the development set; **diagonal, L-shaped and stepped** remain untouched until validation. The original camera pulses are at 2, 4, 6 ms (old) and 40, 42, 44 ms (new); scoring windows are [90,105) and [105,120) ms.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import brian2 as b2

b2.prefs.codegen.target = "numpy"  # Many short simulations: avoid repeated C++ compilation.

# Select which approved behavior to reproduce.
target = "original"  # "fading" (latest approved video model) or "original" (Notebook 20)
assert target in ("fading", "original")

width = height = 9
n = width * height
xy = np.column_stack((np.arange(n) % width, np.arange(n) // width))
dt_ms = 0.05

def pixels(coords):
    return np.array([y * width + x for x, y in coords], dtype=int)

# Exact geometry definitions from Notebook 19 / 20.
shapes = {
    "vertical": (pixels([(2, y) for y in range(2, 7)]),
                 pixels([(5, y) for y in range(2, 7)])),
    "horizontal": (pixels([(x, 2) for x in range(2, 7)]),
                   pixels([(x, 5) for x in range(2, 7)])),
    "diagonal": (pixels([(x, x) for x in range(1, 6)]),
                 pixels([(x + 3, x) for x in range(1, 6)])),
    "L-shaped": (pixels([(2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]),
                 pixels([(5, 2), (5, 3), (5, 4), (6, 4), (7, 4)])),
    "stepped": (pixels([(2, 2), (3, 2), (3, 3), (4, 3), (4, 4)]),
                pixels([(5, 2), (6, 2), (6, 3), (7, 3), (7, 4)])),
}

# Reused, homogeneous surround graph (same for both models and all input shapes).
distance = np.linalg.norm(xy[:, None, :] - xy[None, :, :], axis=-1)
post, pre = np.where((distance > 1.5) & (distance <= 5.5))
base_inhibition = 0.1 * np.exp(-distance[post, pre]**2 / (2 * 2.5**2))
assert all(len(a) == len(b) == 5 and len(set(a) | set(b)) == 10
           for a, b in shapes.values())

fig, axes = plt.subplots(1, 5, figsize=(15, 3.1), constrained_layout=True)
for ax, (name, (old, new)) in zip(axes, shapes.items()):
    image = np.zeros(n, dtype=int)
    image[old], image[new] = 1, 2
    ax.imshow(image.reshape(height, width), cmap="viridis", vmin=0, vmax=2,
              interpolation="nearest")
    ax.set(title=name, xlabel="x (pixel)", ylabel="y (pixel)",
           xticks=range(0, 9, 2), yticks=range(0, 9, 2))
fig.suptitle("Fixed test geometries — old (blue-green), new (yellow)")
plt.show()
print(f"{n} LIF neurons; {len(pre)} directed inhibitory connections; target: {target}")

## 2. Reuse the original Brian2 simulator, changing only autapse and refractory period

The neuron and synaptic currents obey

$$
\tau_m\dot v=-v+I_{\rm ext}+I_{\rm exc}-I_{\rm inh},\qquad
\tau_k\dot I_k=-I_k
$$

with $\tau_m=10$ ms, $\tau_{\rm ext}=3$ ms, $\tau_{\rm exc}=5$ ms and $\tau_{\rm inh}=12$ ms. Camera events add 22 to $I_{\rm ext}$, firing threshold is $v>1$, and reset is $v\leftarrow0$. During refractoriness, **differential equations continue evolving** (the established Brian2 convention).

The two synaptic operations to compare are

$$
\text{frozen: }I_{\rm exc}^+=8,\qquad
\text{additive: }I_{\rm exc}^+=I_{\rm exc}^-+W.
$$

Only the candidate's $W$ and $t_{\rm ref}$ will vary. Inhibition is identical and fixed.

In [ ]:
def simulate(schedule, *, mode="refresh", weight=8., refractory_ms=0.,
             inhibition_scale=1., support=False, dt=dt_ms,
             duration_ms=130., trace_pixel=None):
    b2.start_scope()
    b2.defaultclock.dt = dt * b2.ms

    event_ids, event_times = [], []
    for ids, pulses, stagger_ms in schedule:
        for pulse in pulses:
            for j, pixel in enumerate(ids):
                event_ids.append(int(pixel))
                event_times.append(float(pulse) + j * stagger_ms)
    order = np.argsort(event_times, kind="stable")
    camera = b2.SpikeGeneratorGroup(
        n, np.asarray(event_ids, dtype=int)[order],
        np.asarray(event_times, dtype=float)[order] * b2.ms)

    parameters = dict(tau_m=10*b2.ms, tau_ext=3*b2.ms,
                      tau_exc=5*b2.ms, tau_inh=12*b2.ms,
                      tau_support=120*b2.ms, v_threshold=1., camera_gain=22.)
    equations = """
        dv/dt = (-v + Iext + Iexc - Iinh)/tau_m : 1
        dIext/dt = -Iext/tau_ext : 1
        dIexc/dt = -Iexc/tau_exc : 1
        dIinh/dt = -Iinh/tau_inh : 1
    """
    if support:
        equations += "dsupport/dt = -support/tau_support : 1\n"
    neurons = b2.NeuronGroup(
        n, equations, threshold="v > v_threshold", reset="v = 0",
        refractory=refractory_ms*b2.ms if refractory_ms else False,
        method="exact", namespace=parameters)

    camera_update = "Iext_post += camera_gain"
    if support:
        camera_update += "; support_post = 1"
    feed = b2.Synapses(camera, neurons, on_pre=camera_update, namespace=parameters)
    feed.connect(j="i")

    synaptic_factor = "support_post" if support else "1"
    if mode == "refresh":
        autapse_rule = f"Iexc_post = recurrence_gain * {synaptic_factor}"
    elif mode == "additive":
        autapse_rule = f"Iexc_post += recurrence_gain * {synaptic_factor}"
    else:
        raise ValueError(mode)
    autapse = b2.Synapses(neurons, neurons, on_pre=autapse_rule,
                         namespace={"recurrence_gain": float(weight)})
    autapse.connect(i=np.arange(n), j=np.arange(n))

    inhibition = b2.Synapses(neurons, neurons, "w : 1", on_pre="Iinh_post += w")
    inhibition.connect(i=pre, j=post)
    inhibition.w = base_inhibition * inhibition_scale

    spikes = b2.SpikeMonitor(neurons)
    objects = [camera, neurons, feed, autapse, inhibition, spikes]
    if trace_pixel is not None:
        variables = ["v", "Iext", "Iexc", "Iinh"]
        if support:
            variables.append("support")
        state = b2.StateMonitor(neurons, variables, record=[int(trace_pixel)])
        objects.append(state)

    b2.Network(*objects).run(duration_ms*b2.ms)
    result = {"t": np.asarray(spikes.t/b2.ms), "i": np.asarray(spikes.i, dtype=int)}
    if trace_pixel is not None:
        result["trace_t"] = np.asarray(state.t/b2.ms)
        for var in variables:
            result[var] = np.asarray(getattr(state, var)[0])
    return result

reference = dict(mode="refresh", weight=8., refractory_ms=0.,
                 inhibition_scale=1., support=(target == "fading"))
windows = ((32., 42.), (42., 52.)) if target == "fading" else ((90., 105.), (105., 120.))
second_start_default = 20. if target == "fading" else 40.
old_pulses = (2., 4., 6.)

def pulse_schedule(old, new, second_start, row_delay=0.):
    a = (old, old_pulses, row_delay)
    b = (new, (second_start, second_start+2., second_start+4.), row_delay)
    return a, b

print("Reference:", reference)
print("Scoring windows:", windows)
print("Second stimulus starts at:", second_start_default, "ms")

## 3. Verify the frozen benchmark before calibrating

A shape is **eligible** if every old pixel and every new pixel, when stimulated in isolation, emits at least two spikes in **each** scoring window. For an eligible shape, strict replacement requires all five obsolete pixels silent, all five new pixels active, and no off-target spikes in both windows. This prevents fading or failed recruitment from counting as successful replacement.

The unchanged Notebook 20 benchmark must reproduce **3/5** geometries: vertical, horizontal and L-shaped. We stop via an assertion if it does not.

In [ ]:
def pixel_counts(run, ids, start, stop):
    mask = (run["t"] >= start) & (run["t"] < stop)
    return np.bincount(run["i"][mask], minlength=n)[ids]

def evaluate_one(geometry, settings, *, second_start=second_start_default,
                 row_delay=0., dt=dt_ms):
    old, new = shapes[geometry]
    old_input, new_input = pulse_schedule(old, new, second_start, row_delay)
    kw = dict(settings, dt=dt, duration_ms=130.)
    old_only = simulate([old_input], **kw)
    new_only = simulate([new_input], **kw)
    both = simulate([old_input, new_input], **kw)
    outside = np.setdiff1d(np.arange(n), np.r_[old, new])

    def stable(run, ids):
        return np.array([pixel_counts(run, ids, a, b) for a, b in windows])

    old_control = np.all(stable(old_only, old) >= 2, axis=0)
    new_control = np.all(stable(new_only, new) >= 2, axis=0)
    old_silent = np.all(stable(both, old) == 0, axis=0)
    new_active = np.all(stable(both, new) >= 2, axis=0)
    off_target = np.any(stable(both, outside) > 0, axis=0)
    eligible = bool(old_control.all() and new_control.all())
    return dict(geometry=geometry, old_control=int(old_control.sum()),
                new_control=int(new_control.sum()), eligible=eligible,
                old_silent=int(old_silent.sum()), new_persistent=int(new_active.sum()),
                off_target=int(off_target.sum()),
                success=bool(eligible and old_silent.all() and
                             new_active.all() and not off_target.any()))

def evaluate_geometries(settings, names=tuple(shapes), *, second_start=second_start_default,
                        row_delay=0., dt=dt_ms):
    return pd.DataFrame([evaluate_one(name, settings, second_start=second_start,
                                     row_delay=row_delay, dt=dt) for name in names])

def totals(table):
    return dict(eligible=int(table.eligible.sum()),
                successes=int(table.success.sum()),
                old_silent=float(table.old_silent.mean()),
                new_persistent=float(table.new_persistent.mean()),
                off_target=int(table.off_target.sum()))

reference_results = evaluate_geometries(reference)
print("Reference geometry results:")
display(reference_results)
print("Reference summary:", totals(reference_results))
if target == "original":
    assert reference_results.success.sum() == 3, "Notebook 20 benchmark changed: investigate before calibrating."
    assert set(reference_results.loc[reference_results.success, "geometry"]) == {"vertical", "horizontal", "L-shaped"}
else:
    assert reference_results.eligible.sum() == 5, "Unexpected reference fading: review evaluation windows."


## 4. Derive a starting additive weight from the reference firing interval

If the reference is periodically spiking with interspike interval $T$, it refreshes its excitatory current to 8 after every spike. A purely additive autapse of impulse $W$ with decay time $\tau_{\rm exc}$ has periodic steady-state post-spike current

$$
I_{\rm exc}^{+}=\frac{W}{1-e^{-T/\tau_{\rm exc}}}.
$$

Matching the **post-spike** reference value suggests

$$
\boxed{W_0=8\left(1-e^{-T/\tau_{\rm exc}}\right)}.
$$

This is **not an equivalence theorem**: the additive and refreshed currents differ during transients and inhibitory interruptions. We measure the reference median ISI after recruitment and use $W_0$ only to centre a modest parameter grid. With no support variable, neither network is expected to fade spontaneously in the same way as ESCM 2.0 + sensory fading.

In [ ]:
center = int(pixels([(4, 4)])[0])
solo_input = [(np.array([center]), old_pulses, 0.)]
reference_solo = simulate(solo_input, **reference, duration_ms=140., trace_pixel=center)
reference_spikes = reference_solo["t"]
steady_spikes = reference_spikes[(reference_spikes >= 20.) & (reference_spikes < 110.)]
assert len(steady_spikes) >= 3, "Reference is not persistently spiking; inspect its input and simulation."
T_ms = float(np.median(np.diff(steady_spikes)))
W0 = 8. * (-np.expm1(-T_ms / 5.))
reference_rate_hz = len(steady_spikes) / 0.090
print(f"Reference median ISI [20,110) ms: {T_ms:.3f} ms")
print(f"Reference mean rate [20,110) ms: {reference_rate_hz:.1f} Hz")
print(f"Analytical additive starting weight W0: {W0:.4f}")

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, constrained_layout=True)
axes[0].plot(reference_solo["trace_t"], reference_solo["Iexc"], lw=1.5, label="Reference recurrent current")
axes[0].axhline(8, color="0.4", ls="--", label="Assignment value = 8")
axes[0].set(ylabel="Excitatory current (model units)", title="Frozen ESCM 2.0: isolated autaptic activity")
axes[0].legend()
axes[1].plot(reference_solo["trace_t"], reference_solo["v"], lw=1.2)
axes[1].scatter(reference_spikes, np.ones_like(reference_spikes), marker="|", s=65, label="Output spikes")
axes[1].axhline(1, color="0.4", ls="--", label="Threshold")
axes[1].set(xlabel="Time (ms)", ylabel="Membrane voltage (model units)")
axes[1].legend()
plt.show()

## 5. Calibrate $W$ and refractory period on two development geometries

Test additive weights as multiples of the analytically motivated $W_0$ and refractory periods of 0, 0.5, 1 and 2 ms. **Do not tune inhibition**. For every candidate, measure isolated-neuron firing and the strict spatial outcome on vertical and horizontal patterns using old-only, new-only, and combined inputs.

Rank candidates by number of **eligible** development geometries, then strict successes, then isolated firing-rate proximity to the reference. A high spatial score with failed isolated controls cannot be accepted. Choosing a single winner on the development set is a hypothesis for validation, not evidence of generalization.

The search is finite and targeted; failure does not rule out all possible additive LIF parameters.

In [ ]:
development = ("vertical", "horizontal")
held_out = ("diagonal", "L-shaped", "stepped")
weights = W0 * np.array([0.5, 0.75, 1., 1.25, 1.5, 2., 3., 4.])
refractory_periods_ms = (0., 0.5, 1., 2.)

rows = []
for refractory_ms in tqdm(refractory_periods_ms, desc="Calibrate additive autapse"):
    for W in weights:
        settings = dict(mode="additive", weight=float(W), refractory_ms=refractory_ms,
                        inhibition_scale=1., support=False)
        run = simulate(solo_input, **settings, duration_ms=140.)
        rate_hz = np.count_nonzero((run["t"] >= 20.) & (run["t"] < 110.)) / 0.090
        table = evaluate_geometries(settings, development)
        rows.append(dict(W=float(W), refractory_ms=refractory_ms,
                         solo_rate_hz=rate_hz,
                         rate_error_hz=abs(rate_hz-reference_rate_hz),
                         **totals(table)))
search_table = pd.DataFrame(rows)
search_table = search_table.sort_values(
    ["eligible", "successes", "rate_error_hz", "old_silent", "new_persistent", "off_target"],
    ascending=[False, False, True, False, False, True]).reset_index(drop=True)
display(search_table.round(3))
winner = search_table.iloc[0]
candidate = dict(mode="additive", weight=float(winner.W),
                 refractory_ms=float(winner.refractory_ms),
                 inhibition_scale=1., support=False)
print("Selected development-only additive candidate:", candidate)
print("Development-only candidate results:")
display(evaluate_geometries(candidate, development))

## 6. Freeze the candidate and test spatial generalization

Evaluate the frozen candidate on **all five** original geometries, without modifying any parameters. The diagonal, L-shaped and stepped cases were not involved in selecting (W) or the refractory period. Report eligible controls, obsolete pixels silenced, new pixels retained, off-target activity, and strict success for each model.

The benchmark's full result is 3/5; a candidate that matches development but fails the held-out L shape **does not match ESCM 2.0 spatially**. Treat exact per-geometry agreement as more informative than an equal aggregate count achieved on different cases.

In [ ]:
candidate_results = evaluate_geometries(candidate)
comparison = reference_results.merge(candidate_results, on="geometry", suffixes=("_ref", "_add"))
display(comparison)
print("Frozen reference:", totals(reference_results))
print("Additive candidate:", totals(candidate_results))
print("Held-out reference successes:", int(reference_results.set_index("geometry").loc[list(held_out), "success"].sum()))
print("Held-out additive successes:", int(candidate_results.set_index("geometry").loc[list(held_out), "success"].sum()))

fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
x = np.arange(len(comparison))
ax.bar(x - 0.2, comparison["old_silent_ref"], width=0.38, label="ESCM 2.0: old pixels silent")
ax.bar(x + 0.2, comparison["old_silent_add"], width=0.38, label="Additive: old pixels silent")
ax.scatter(x - 0.2, comparison["new_persistent_ref"], color="black", marker="o",
           label="ESCM 2.0: new pixels active")
ax.scatter(x + 0.2, comparison["new_persistent_add"], color="black", marker="D",
           label="Additive: new pixels active")
ax.axhline(5, color="0.4", ls="--")
ax.set(xticks=x, xticklabels=comparison["geometry"], ylim=(0, 5.6),
       ylabel="Pixels satisfying both evaluation windows",
       title="Frozen spatial replacement: suppressed old / retained new")
ax.legend(loc="lower left", ncol=2)
plt.show()

## 7. Test the original asynchronous timing challenges

The original non-fading benchmark allows a common **absolute** evaluation window [90,105) and [105,120) ms even when the second stimulus starts at 20, 40, 60 or 80 ms. Evaluate all four onsets crossed with pixel staggering of 0, 0.25, 0.5 and 1 ms on the development vertical geometry. These timing conditions test robustness; they are not independent held-out geometries.

In [ ]:
timing_rows = []
for onset in tqdm((20., 40., 60., 80.), desc="Timing challenges"):
    for stagger in (0., 0.25, 0.5, 1.):
        for name, settings in (("Frozen ESCM 2.0", reference), ("Additive", candidate)):
            timing_rows.append(dict(model=name, onset_ms=onset, stagger_ms=stagger,
                                    **evaluate_one("vertical", settings,
                                                   second_start=onset, row_delay=stagger)))
timing = pd.DataFrame(timing_rows)
display(timing[["model", "onset_ms", "stagger_ms", "eligible", "success",
                "old_silent", "new_persistent", "off_target"]])
display(timing.groupby("model")[["eligible", "success"]].sum())

## 8. Inspect the current balance during inhibition

For successful and unsuccessful geometries, a single strict score cannot reveal whether the old neurons survive due to low inhibition or excessive recurring excitation. Record representative old and new neurons under **identical** input in the two models. Plot $I_{\rm exc}$, $I_{\rm inh}$, $I_{\rm ext}+I_{\rm exc}-I_{\rm inh}$, and spike rasters. Net current is a membrane forcing term, not a direct prediction of firing because voltage integrates over time and has resets/refractoriness.

We inspect vertical (development) and L-shaped (held-out) cases without altering the simulator or its parameters.

In [ ]:
for geometry in ("vertical", "L-shaped"):
    old, new = shapes[geometry]
    schedule = pulse_schedule(old, new, second_start_default)
    monitored = {}
    for name, settings in (("ESCM 2.0", reference), ("Additive", candidate)):
        monitored[name] = (simulate(schedule, **settings, duration_ms=125.,
                                    trace_pixel=int(old[2])),
                           simulate(schedule, **settings, duration_ms=125.,
                                    trace_pixel=int(new[2])))
    fig, axes = plt.subplots(4, 2, figsize=(13, 10), sharex=True, constrained_layout=True)
    for col, (name, (old_run, new_run)) in enumerate(monitored.items()):
        for run, group, style in ((old_run, "old", "-"), (new_run, "new", "--")):
            t = run["trace_t"]
            axes[0, col].plot(t, run["Iexc"], ls=style, label=group)
            axes[1, col].plot(t, run["Iinh"], ls=style, label=group)
            axes[2, col].plot(t, run["Iext"] + run["Iexc"] - run["Iinh"], ls=style, label=group)
            times = run["t"][run["i"] == (old[2] if group == "old" else new[2])]
            axes[3, col].vlines(times, 0 if group == "old" else 1,
                                0.7 if group == "old" else 1.7, lw=1)
        axes[0, col].set_title(name)
        axes[0, col].legend()
        axes[3, col].set(xlabel="Time (ms)", yticks=(0.35, 1.35),
                         yticklabels=("old", "new"))
        for row in range(4):
            axes[row, col].axvline(second_start_default, color="0.5", ls=":", lw=1)
            axes[row, col].set_xlim(0, 125)
    for row, label in enumerate(("Excitatory current", "Inhibitory current",
                                 "Net driving current", "Spike times")):
        axes[row, 0].set_ylabel(label + (" (model units)" if row < 3 else ""))
    fig.suptitle(f"{geometry}: current balance and competing activity")
    plt.show()

## 9. Numerical robustness without retuning

Repeat the five original geometries at timesteps 0.1, 0.05 and 0.025 ms for both **fixed** models. The required observation is stable **classification**, not identical spike timestamps. Disagreement with the reference at any resolution is evidence of a different network behavior. Timestep stability over these three values is not a proof of convergence.

In [ ]:
resolution_rows = []
for dt in tqdm((0.1, 0.05, 0.025), desc="Timestep robustness"):
    for name, settings in (("Frozen ESCM 2.0", reference), ("Additive", candidate)):
        for geometry in shapes:
            resolution_rows.append(dict(dt_ms=dt, model=name,
                                        **evaluate_one(geometry, settings, dt=dt)))
resolution = pd.DataFrame(resolution_rows)
display(resolution[["dt_ms", "model", "geometry", "eligible",
                    "success", "old_silent", "new_persistent"]])
display(resolution.groupby(["model", "dt_ms"])[["eligible", "success"]].sum())

## 10. Interpretation and the gate to SRR

The Stage A result should be judged against **all** the following observations, not simply the best development score:

- The unchanged frozen reference reproduces its established 3/5 geometry benchmark.
- The additive candidate supports persistent autonomous activity with a reasonable refractory-limited rate and remains eligible in isolated controls.
- The candidate reproduces the reference's strict vertical, horizontal and L-shaped replacement outcomes, rather than merely reaching a matching total on different geometries.
- The 16 timing challenges and three timestep classifications do not expose material regressions.
- The current traces explain any remaining discrepancy; changing autaptic parameters is allowed, but inhibition has been held fixed for this first attempt.

**If all criteria hold:** freeze the candidate and begin Stage B, adding camera-replenished resources to the additive autapse. Compare that SRR model with the **separate ESCM 2.0 + fading** reference using its appropriate stimulus-relative windows.

**If they do not hold:** retain this notebook as a negative or partial result. Diagnose whether the limitation is self-excitation/refractoriness or susceptibility to the fixed inhibitory graph before expanding the calibration. Do **not** use the held-out results to repeatedly tune parameters while continuing to call them held out.

**Execution status:** cells are intentionally unexecuted in this committed notebook. No success is claimed until Brian2 actually reproduces the outputs.